# Exploratory Data Analysis — SQL

**IBM Data Science Capstone — SpaceX Falcon 9**

Repository: [https://github.com/MCerros/IBM-Data-Science-Capstone-SpaceX](https://github.com/MCerros/IBM-Data-Science-Capstone-SpaceX)

Direct notebook URL after upload:  
[https://github.com/MCerros/IBM-Data-Science-Capstone-SpaceX/blob/main/05_SpaceX_EDA_SQL.ipynb](https://github.com/MCerros/IBM-Data-Science-Capstone-SpaceX/blob/main/05_SpaceX_EDA_SQL.ipynb)

**Data integrity note:** This notebook uses the project CSV files stored in the same repository.
No rows, metrics, charts, or model scores are manually invented.

## Objective

Load `Spacex.csv` into an in-memory SQLite database and execute the SQL analyses
required by the capstone: launch sites, payload statistics, landing outcomes,
booster versions, classifications, success rates, and temporal analysis.

In [1]:
import sqlite3
import pandas as pd

spacex = pd.read_csv("Spacex.csv")
conn = sqlite3.connect(":memory:")
spacex.to_sql("SPACEXTBL", conn, index=False, if_exists="replace")

def sql(query):
    return pd.read_sql_query(query, conn)

print("Rows loaded into SPACEXTBL:", len(spacex))

Rows loaded into SPACEXTBL: 101


### 1. Unique launch sites

In [2]:
sql("SELECT DISTINCT Launch_Site FROM SPACEXTBL;")

,Launch_Site
0,CCAFS LC-40
1,VAFB SLC-4E
2,KSC LC-39A
3,CCAFS SLC-40


### 2. Five records with launch site names beginning with CCA

In [3]:
sql("SELECT * FROM SPACEXTBL WHERE Launch_Site LIKE 'CCA%' LIMIT 5;")

,Date,Time (UTC),Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing_Outcome
0,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
1,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of...",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
3,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
4,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt


### 3. Total payload mass carried for NASA (CRS)

In [4]:
sql("SELECT SUM(PAYLOAD_MASS__KG_) AS Total_Payload_kg FROM SPACEXTBL WHERE Customer='NASA (CRS)';")

,Total_Payload_kg
0,45596


### 4. Average payload mass for Falcon 9 v1.1

In [5]:
sql("SELECT AVG(PAYLOAD_MASS__KG_) AS Average_Payload_kg FROM SPACEXTBL WHERE Booster_Version LIKE 'F9 v1.1%';")

,Average_Payload_kg
0,2534.666667


### 5. Date of first successful ground-pad landing

In [6]:
sql("SELECT MIN(Date) AS First_Successful_Ground_Landing FROM SPACEXTBL WHERE Landing_Outcome='Success (ground pad)';")

,First_Successful_Ground_Landing
0,2015-12-22


### 6. Successful drone-ship landings with payload between 4,000 and 6,000 kg

In [7]:
sql("SELECT Booster_Version, PAYLOAD_MASS__KG_, Landing_Outcome FROM SPACEXTBL WHERE Landing_Outcome='Success (drone ship)' AND PAYLOAD_MASS__KG_ BETWEEN 4000 AND 6000;")

,Booster_Version,PAYLOAD_MASS__KG_,Landing_Outcome
0,F9 FT B1022,4696,Success (drone ship)
1,F9 FT B1026,4600,Success (drone ship)
2,F9 FT B1021.2,5300,Success (drone ship)
3,F9 FT B1031.2,5200,Success (drone ship)


### 7. Mission outcome counts

In [8]:
sql("SELECT Mission_Outcome, COUNT(*) AS Count FROM SPACEXTBL GROUP BY Mission_Outcome ORDER BY Count DESC;")

,Mission_Outcome,Count
0,Success,98
1,Success (payload status unclear),1
2,Success,1
3,Failure (in flight),1


### 8. Booster versions that carried the maximum payload

In [9]:
sql("SELECT Booster_Version, PAYLOAD_MASS__KG_ FROM SPACEXTBL WHERE PAYLOAD_MASS__KG_=(SELECT MAX(PAYLOAD_MASS__KG_) FROM SPACEXTBL);")

,Booster_Version,PAYLOAD_MASS__KG_
0,F9 B5 B1048.4,15600
1,F9 B5 B1049.4,15600
2,F9 B5 B1051.3,15600
3,F9 B5 B1056.4,15600
4,F9 B5 B1048.5,15600
5,F9 B5 B1051.4,15600
6,F9 B5 B1049.5,15600
7,F9 B5 B1060.2,15600
8,F9 B5 B1058.3,15600
9,F9 B5 B1051.6,15600


### 9. Failed drone-ship landings in 2015

In [10]:
sql("SELECT strftime('%m',Date) AS Month, Landing_Outcome, Booster_Version, Launch_Site FROM SPACEXTBL WHERE strftime('%Y',Date)='2015' AND Landing_Outcome='Failure (drone ship)';")

,Month,Landing_Outcome,Booster_Version,Launch_Site
0,01,Failure (drone ship),F9 v1.1 B1012,CCAFS LC-40
1,04,Failure (drone ship),F9 v1.1 B1015,CCAFS LC-40


### 10. Distribution of landing outcomes from 2010-06-04 to 2017-03-20

In [11]:
sql("SELECT Landing_Outcome, COUNT(*) AS Count FROM SPACEXTBL WHERE Date BETWEEN '2010-06-04' AND '2017-03-20' GROUP BY Landing_Outcome ORDER BY Count DESC;")

,Landing_Outcome,Count
0,No attempt,10
1,Success (drone ship),5
2,Failure (drone ship),5
3,Success (ground pad),3
4,Controlled (ocean),3
5,Uncontrolled (ocean),2
6,Failure (parachute),2
7,Precluded (drone ship),1


### 11. Landing success rate by launch site

In [12]:
sql('''
SELECT
    Launch_Site,
    COUNT(*) AS Launches,
    SUM(CASE WHEN Landing_Outcome LIKE 'Success%' THEN 1 ELSE 0 END) AS Successful_Landings,
    ROUND(
        1.0 * SUM(CASE WHEN Landing_Outcome LIKE 'Success%' THEN 1 ELSE 0 END) / COUNT(*),
        4
    ) AS Landing_Success_Rate
FROM SPACEXTBL
GROUP BY Launch_Site
ORDER BY Landing_Success_Rate DESC;
''')

,Launch_Site,Launches,Successful_Landings,Landing_Success_Rate
0,KSC LC-39A,25,20,0.8000
1,CCAFS SLC-40,34,25,0.7353
2,VAFB SLC-4E,16,10,0.6250
3,CCAFS LC-40,26,6,0.2308
